# 13.4 · DCGAN 与 WGAN / DCGAN & WGAN

> **课程定位 / Where this fits**
> 第 4 课，**Part 13 · 生成模型**。针对 GAN 训练不稳定的两大经典改进。
> Lesson 4, **Part 13 · Generative Models**. Two classic fixes for GAN's instability.
>
> 13.3 的 GAN 用全连接(MLP)且训练不稳。两个里程碑式改进各从一个角度入手：**DCGAN** 改**结构**——用卷积(转置卷积上采样)替代全连接, 生成更清晰的图, 并给出一套稳定训练的架构准则；**WGAN** 改**损失**——用 **Wasserstein(推土机)距离**替代原始的 JS 散度, 让损失**有意义且与生成质量相关**、缓解梯度消失和模式坍塌。本课讲清两者的核心思想, 搭出 DCGAN 卷积结构, 并**从零实现 WGAN** 看其"损失可监控训练"的优势。
> The GAN in 13.3 used fully-connected layers and trained unstably. Two landmark fixes each target one angle: **DCGAN** changes the **architecture** — convolutions (transposed conv for upsampling) instead of FC, for crisper images, with guidelines for stable training; **WGAN** changes the **loss** — the **Wasserstein (earth-mover) distance** instead of the original JS divergence, making the loss **meaningful and correlated with quality**, easing vanishing gradients and mode collapse. We explain both, build the DCGAN conv architecture, and **implement WGAN from scratch** to see its "loss tracks training" advantage.
>
> 💼 **实战/面试视角**："DCGAN 架构准则 / 转置卷积 / Wasserstein 距离为什么更好 / 权重裁剪 vs 梯度惩罚(WGAN-GP) / critic vs discriminator" 是 GAN 进阶常考。
> 💼 **Practical/interview angle:** "DCGAN guidelines / transposed conv / why Wasserstein / weight clipping vs gradient penalty (WGAN-GP) / critic vs discriminator" — advanced GAN questions.

> 📐 **符号约定 / Notation**
> - 转置卷积 —— 可学习的上采样(放大特征图) / transposed conv: learnable upsampling
> - critic —— WGAN 的"判别器", 输出**实数分数**而非概率 / outputs a real score, not a probability
> - W 距离 —— Wasserstein / Earth-Mover 距离 / Wasserstein distance

> 💡 **面试相关 / Interview-relevant**
> - "DCGAN 的架构设计准则"（出镜率 ★★★★）
> - "转置卷积怎么上采样"（★★★）
> - "WGAN 为什么比原始 GAN 稳定"（出镜率 ★★★★★）
> - "权重裁剪 vs 梯度惩罚(WGAN-GP)"（★★★★）
> - "为什么 WGAN 的损失有意义"（★★★★）

---

## 学习目标 / Learning Objectives
1. 理解 DCGAN 的卷积架构与稳定训练准则。
   Understand DCGAN's conv architecture and stability guidelines.
2. 会用**转置卷积**搭生成器(上采样到图像)。
   Build a generator with transposed convolutions.
3. 理解 **Wasserstein 距离**为何让 GAN 更稳。
   Understand why the Wasserstein distance stabilizes GANs.
4. **从零实现 WGAN**, 看其损失可监控训练质量。
   Implement WGAN from scratch; see its loss track quality.

## 目录 / TOC
1. [GAN 不稳定的两条改进路 ⭐](#1)
2. [DCGAN：卷积架构 ⭐](#2)
3. [WGAN：Wasserstein 损失（从零）⭐](#3)
4. [权重裁剪 vs WGAN-GP + 小结 ⭐](#4)


<a id="1"></a>
## 1. GAN 不稳定的两条改进路 ⭐ / Two Routes to Fix GAN Instability

13.3 暴露了 GAN 的两大痛点：训练不稳(损失震荡、不收敛)、模式坍塌。改进从两个**正交**的角度入手(面试可对比)：
13.3 exposed GAN's pains: unstable training (oscillating loss, non-convergence) and mode collapse. Fixes come from two **orthogonal** angles:
- **改架构(DCGAN, 2015)**：原始 GAN 用全连接, 不擅长图像。DCGAN 用**卷积/转置卷积**+一套经验准则, 让生成器/判别器更适合图像, 训练更稳、生成更清晰。
  **Architecture (DCGAN, 2015):** the original used FC, poor for images. DCGAN uses **conv/transposed-conv** + a set of empirical guidelines, better-suited to images, training more stably and generating sharper.
- **改损失(WGAN, 2017)**：原始 GAN 的损失等价于最小化 JS 散度——当真假分布几乎不重叠时(训练初期常见), JS 散度**梯度几乎为 0**, 生成器学不动。WGAN 换成 **Wasserstein 距离**, 它**即使分布不重叠也有平滑有意义的梯度**, 从根上改善训练。
  **Loss (WGAN, 2017):** the original loss minimizes JS divergence — when real/fake barely overlap (common early), JS gives **near-zero gradients** and the generator stalls. WGAN uses the **Wasserstein distance**, which gives **smooth meaningful gradients even with non-overlapping distributions**, fixing training at the root.

两者**可以叠加**(DCGAN 架构 + WGAN 损失), 实践中常一起用。
The two **compose** (DCGAN architecture + WGAN loss) and are often combined in practice.


<a id="2"></a>
## 2. DCGAN：卷积架构 ⭐ / DCGAN: Convolutional Architecture

**DCGAN(深度卷积 GAN)** 把 13.3 的全连接换成卷积。核心是**生成器用转置卷积逐步上采样**：从一个噪声向量 → reshape 成小特征图 → 一层层**转置卷积放大**(7×7→14×14→28×28), 最后输出图像。判别器则用**普通卷积逐步下采样**(和分类 CNN 一样)。
**DCGAN** replaces 13.3's FC with conv. The generator **upsamples step by step with transposed convolutions**: noise → reshape to a small feature map → **transposed convs enlarge** it (7×7→14×14→28×28) → output image. The discriminator **downsamples with normal convs** (like a classification CNN).

**转置卷积(transposed conv)**：可看作卷积的"逆操作"——把小特征图**放大**(每个像素扩展成一块), 是**可学习的上采样**(对比固定的插值)。
**Transposed conv:** the "inverse" of convolution — **enlarges** a small feature map (each pixel expands to a patch); a **learnable upsampling** (vs fixed interpolation).

**DCGAN 经验准则(面试常考)**：①用**步幅卷积/转置卷积**替代池化(让网络自己学上/下采样)；②生成器和判别器都用 **BatchNorm**(稳定训练)；③去掉全连接隐层；④生成器用 **ReLU**(输出层 Tanh)、判别器用 **LeakyReLU**。
**DCGAN guidelines:** ① use **strided (transposed) conv** instead of pooling; ② **BatchNorm** in both G and D; ③ no FC hidden layers; ④ ReLU in G (Tanh output), LeakyReLU in D.

下面搭出 DCGAN 的卷积生成器和判别器, 验证形状(它在 GPU 上训练能生成清晰图像; 本课重点是结构, 完整训练较慢)。
Below we build DCGAN's conv generator and discriminator and verify shapes (trained on GPU it yields crisp images; here we focus on architecture, as full conv training is slow on CPU).


In [ ]:
import os, numpy as np, matplotlib.pyplot as plt, seaborn as sns, time
import torch, torch.nn as nn
sns.set_theme(style="white"); torch.manual_seed(0)
Z = 64

class DCGenerator(nn.Module):
    """DCGAN 生成器: 噪声 → 转置卷积逐步上采样 → 28×28 图 / noise → transposed conv upsampling → image."""
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(Z, 128*7*7)                  # 噪声投影成 128×7×7 小特征图 / project to a small map
        self.net = nn.Sequential(
            nn.BatchNorm2d(128), nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(),  # 7→14 上采样 / upsample
            nn.ConvTranspose2d(64, 1, 4, stride=2, padding=1), nn.Tanh())                        # 14→28, 输出图 / output
    def forward(self, z): return self.net(self.fc(z).view(-1,128,7,7))

class DCDiscriminator(nn.Module):
    """DCGAN 判别器: 普通卷积逐步下采样 → 真假分数 / strided conv downsampling → real/fake score."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 64, 4, stride=2, padding=1), nn.LeakyReLU(0.2),                         # 28→14 下采样 / downsample
            nn.Conv2d(64, 128, 4, stride=2, padding=1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2),  # 14→7
            nn.Flatten(), nn.Linear(128*7*7, 1))
    def forward(self, x): return self.net(x)

G, D = DCGenerator(), DCDiscriminator()
z = torch.randn(4, Z); fake = G(z)
print(f"DC生成器: 噪声 {tuple(z.shape)} → 转置卷积上采样 → 图像 {tuple(fake.shape)} (28×28)")
print(f"DC判别器: 图像 {tuple(fake.shape)} → 卷积下采样 → 分数 {tuple(D(fake).shape)}")
print(f"生成器参数 {sum(p.numel() for p in G.parameters()):,}, 判别器参数 {sum(p.numel() for p in D.parameters()):,}")
print("\nDCGAN准则: 转置卷积上采样(替代池化) + BatchNorm + 无FC隐层 + G用ReLU(输出Tanh)/D用LeakyReLU")
print("卷积结构利用图像的空间局部性(对比MLP把图拉平) → 生成更清晰; GPU上训练可得锐利图像")


<a id="3"></a>
## 3. WGAN：Wasserstein 损失（从零）⭐ / WGAN: Wasserstein Loss

**为什么原始 GAN 训练难**(面试核心)：它的损失本质是最小化真假分布的 **JS 散度**。但高维图像里, 真实分布和生成分布**几乎不重叠**, 此时 JS 散度是个常数(梯度≈0)——判别器一旦变强, 生成器就**收不到有用梯度**, 训练卡住。
**Why original GANs are hard** (interview core): the loss effectively minimizes the **JS divergence** between real and fake. But for high-dim images the two barely **overlap**, where JS divergence is constant (gradient ≈ 0) — once the discriminator gets strong, the generator gets **no useful gradient** and stalls.

**WGAN** 改用 **Wasserstein(推土机)距离**：直观理解为"把一个分布的'土'搬成另一个分布所需的最小搬运代价"。它的关键好处：**即使两个分布完全不重叠, 也能给出平滑、有意义、不消失的梯度**。
**WGAN** uses the **Wasserstein (earth-mover) distance**: intuitively "the minimum cost to move the 'dirt' of one distribution into the other." Its key benefit: **even when distributions don't overlap, it gives smooth, meaningful, non-vanishing gradients.**

实现上的三个改动(面试)：
Three implementation changes (interview):
1. **判别器变"critic(评论家)"**：输出一个**实数分数**(衡量"有多像真"), 不再是 0~1 概率(去掉 sigmoid)。
   **Discriminator → "critic":** outputs a **real-valued score** (how "real-like"), not a 0–1 probability (drop the sigmoid).
2. **损失 = $\mathbb{E}[\text{critic}(真)] - \mathbb{E}[\text{critic}(假)]$**:critic 想最大化这个差(给真高分给假低分), 生成器想让假图的 critic 分数变高。这个差本身就**估计了 Wasserstein 距离**——所以 **损失值能反映生成质量**(原始 GAN 损失则毫无意义)。
   **Loss = $\mathbb{E}[\text{critic}(\text{real})] - \mathbb{E}[\text{critic}(\text{fake})]$:** the critic maximizes this gap; the generator raises fakes' scores. The gap itself **estimates the Wasserstein distance** — so **the loss value reflects sample quality** (unlike the meaningless original GAN loss).
3. **约束 critic 为 1-Lipschitz**:理论要求 critic 不能"太陡"。原始 WGAN 用**权重裁剪(weight clipping)** 把权重限制在 $[-c, c]$。
   **Constrain the critic to be 1-Lipschitz:** the critic can't be "too steep." Original WGAN uses **weight clipping** to $[-c, c]$.

下面从零实现 WGAN, 重点看**损失(W距离估计)随训练下降**——这是原始 GAN 给不了的"可监控信号"。
We implement WGAN from scratch, focusing on **the loss (W-distance estimate) decreasing during training** — a monitorable signal the original GAN lacks.


In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
DATA_ROOT = os.path.expanduser("~/.cache/dsfs_cv")
tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,),(0.5,))])
loader = DataLoader(Subset(datasets.MNIST(DATA_ROOT, train=True, download=True, transform=tfm), range(15000)),
                    batch_size=128, shuffle=True, drop_last=True)

torch.manual_seed(0)
Gw = nn.Sequential(nn.Linear(Z,256), nn.LeakyReLU(0.2), nn.Linear(256,512), nn.LeakyReLU(0.2), nn.Linear(512,784), nn.Tanh())
# critic: 输出实数分数(无 sigmoid) / critic outputs a real score (no sigmoid)
critic = nn.Sequential(nn.Flatten(), nn.Linear(784,512), nn.LeakyReLU(0.2), nn.Linear(512,256), nn.LeakyReLU(0.2), nn.Linear(256,1))
og = torch.optim.RMSprop(Gw.parameters(), 5e-5)          # WGAN 论文用 RMSprop(无动量) / RMSprop, no momentum
oc = torch.optim.RMSprop(critic.parameters(), 5e-5)
n_critic, clip = 5, 0.01                                  # critic 多训几次; 权重裁剪到[-0.01,0.01] / critic steps; clip
w_curve = []; t0 = time.time()
for ep in range(20):
    for i, (xb, _) in enumerate(loader):
        b = xb.size(0); fake = Gw(torch.randn(b, Z)).view(-1,1,28,28)
        # --- 训 critic: 最大化 E[critic(真)]-E[critic(假)] / train critic ---
        oc.zero_grad()
        loss_c = -(critic(xb).mean() - critic(fake.detach()).mean())     # 取负→最小化 / negate to minimize
        loss_c.backward(); oc.step()
        for p in critic.parameters(): p.data.clamp_(-clip, clip)         # 权重裁剪(1-Lipschitz约束) / weight clipping
        # --- 每 n_critic 步训一次生成器 / train generator every n_critic steps ---
        if i % n_critic == 0:
            fake = Gw(torch.randn(b, Z)).view(-1,1,28,28)
            og.zero_grad(); loss_g = -critic(fake).mean(); loss_g.backward(); og.step()   # 让假图分数变高 / raise fake scores
    with torch.no_grad():
        w = (critic(xb).mean() - critic(Gw(torch.randn(128,Z)).view(-1,1,28,28)).mean()).item()  # W距离估计 / W estimate
    w_curve.append(w)
print(f"WGAN 训练完成 ({time.time()-t0:.0f}s)")
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].plot(w_curve, "o-"); axes[0].set_xlabel("epoch"); axes[0].set_ylabel("Wasserstein 距离估计")
axes[0].set_title("WGAN 损失=W距离估计, 随训练下降(=可监控生成质量!)")
with torch.no_grad(): samp = Gw(torch.randn(8, Z)).view(-1,28,28)
for i in range(8):
    ax = axes[1].inset_axes([i*0.125, 0, 0.12, 1]); ax.imshow(samp[i], cmap="gray"); ax.axis("off")
axes[1].axis("off"); axes[1].set_title("WGAN 生成样本(小MLP/CPU下较粗糙)")
plt.tight_layout(); plt.show()
print(f"W距离估计 {w_curve[0]:.2f} → {w_curve[-1]:.2f}: 损失值随训练下降, 与生成质量相关")
print("⚠️ 这是WGAN最大优点: 损失有意义、可监控(原始GAN的D/G损失震荡无法判断好坏)")
print("⚠️ 诚实说明: 权重裁剪较粗糙, 加上小MLP/CPU+少步数, 样本质量/多样性有限; WGAN-GP(下面)更好")


<a id="4"></a>
## 4. 权重裁剪 vs WGAN-GP + 小结 ⭐ / Weight Clipping vs WGAN-GP

原始 WGAN 用**权重裁剪**强制 critic 是 1-Lipschitz, 但这招**很粗糙**(面试点)：裁剪阈值难调——太小→梯度消失、网络容量被浪费, 太大→又失去约束；还容易让权重都挤到 $\pm c$ 两端。这正是我们上面样本质量有限的原因之一。
Original WGAN uses **weight clipping** for 1-Lipschitz, but it's **crude** (interview): the threshold is hard to tune — too small → vanishing gradients/wasted capacity, too large → loses the constraint; weights tend to pile at $\pm c$. Part of why our samples are limited.

**WGAN-GP(梯度惩罚, 2017)**：改用更优雅的方式约束 Lipschitz——不裁剪权重, 而是**加一个惩罚项, 让 critic 的梯度范数接近 1**(在真假样本的插值点上)。WGAN-GP **训练更稳、生成质量更高**, 是实践中常用的版本。
**WGAN-GP (gradient penalty, 2017):** a more elegant Lipschitz constraint — instead of clipping, **add a penalty pushing the critic's gradient norm toward 1** (at interpolated points). WGAN-GP trains **more stably and generates better**, the common practical version.

**其它稳定 GAN 的技巧**：谱归一化(spectral norm)、TTR(两时间尺度)、自注意力(SAGAN)、渐进式增长(ProGAN)、StyleGAN 等。GAN 曾是图像生成的霸主, 但近年**扩散模型(13.7)** 在质量和稳定性上超越了它。
**Other stabilizers:** spectral normalization, two-time-scale updates (TTUR), self-attention (SAGAN), progressive growing (ProGAN), StyleGAN, etc. GANs once dominated image generation, but recently **diffusion models (13.7)** surpassed them in quality and stability.

```
两条改进路: DCGAN改架构(卷积) + WGAN改损失(Wasserstein); 可叠加
DCGAN: 转置卷积上采样(替代池化)+BatchNorm+无FC隐层+G用ReLU(Tanh输出)/D用LeakyReLU; 利用空间结构→更清晰
WGAN: 原始GAN等价最小化JS散度, 分布不重叠时梯度≈0卡住; Wasserstein距离即使不重叠也有平滑梯度
WGAN实现: 判别器→critic(输出实数分数,无sigmoid); 损失=E[critic(真)]-E[critic(假)](=W距离估计, 可监控质量); critic需1-Lipschitz
1-Lipschitz约束: 权重裁剪(原始WGAN,粗糙) vs 梯度惩罚WGAN-GP(更优雅更稳, 常用)
GAN曾是图像生成霸主, 现被扩散模型(13.7)在质量/稳定性上超越
```

### 💡 面试速查 / Interview cheat-sheet
1. **DCGAN**: 全卷积(转置卷积上采样)+BatchNorm+无FC; 适合图像, 更清晰。
   DCGAN: all-conv (transposed-conv upsampling)+BN+no FC; image-friendly, sharper.
2. **WGAN 为何稳**: Wasserstein距离即使分布不重叠也有有意义梯度(JS散度则≈0)。
   Why WGAN stabilizes: Wasserstein gives meaningful gradients even with non-overlapping distributions.
3. **critic vs discriminator**: 输出实数分数(无sigmoid); 损失=W距离估计可监控质量。
   Critic vs discriminator: real-valued score (no sigmoid); loss = W-distance, tracks quality.
4. **1-Lipschitz**: 权重裁剪(粗糙) vs 梯度惩罚WGAN-GP(更稳更好)。
   1-Lipschitz: weight clipping (crude) vs gradient penalty WGAN-GP (better).
5. **演进**: DCGAN/WGAN/SN/StyleGAN…; 现被扩散模型超越。
   Evolution: DCGAN/WGAN/SN/StyleGAN…; now surpassed by diffusion.

### 下一节 / Next
**13.5 条件 GAN**——前面 GAN 只能"随机生成数字", 不能指定"我要一个 7"。**条件 GAN(cGAN)** 把**标签/条件**喂给生成器和判别器, 实现**可控生成**——指定要哪个数字。这也是 pix2pix(边缘→照片)、文生图等可控生成的基础。
**13.5 Conditional GAN** — so far GANs generate random digits, can't request "a 7." A **conditional GAN (cGAN)** feeds a **label/condition** to both G and D for **controllable generation** — specify which digit. The basis for pix2pix (edges→photo) and text-to-image.
